In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential

# 1. Load Data
data = pd.read_csv('Life Expectancy Data.csv')
data.columns = data.columns.str.strip()

# Drop rows missing the target variable
data = data.dropna(subset=['Life expectancy'])

# 2. Advanced Imputation: Group by Country median first, then global median
num_cols = data.select_dtypes(include=[np.number]).columns.drop('Life expectancy')
for col in num_cols:
    data[col] = data.groupby('Country')[col].transform(lambda x: x.fillna(x.median()))
    data[col] = data[col].fillna(data[col].median())

# 3. Feature Engineering: Key interaction features
data['Schooling_x_Income'] = data['Schooling'] * data['Income composition of resources']
data['Adult_x_HIV'] = data['Adult Mortality'] * data['HIV/AIDS']

# 4. Feature and Target Separation + One-Hot Encoding
X = data.drop(columns=['Life expectancy'])
y = data['Life expectancy']

X = pd.get_dummies(X, columns=['Country', 'Status'], drop_first=True)

# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Scaling Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Build Deep Neural Network Architecture
tf.random.set_seed(42)
np.random.seed(42)

model = Sequential([
    Dense(512, activation='swish', input_dim=X_train_scaled.shape[1]),
    BatchNormalization(),
    Dropout(0.2),

    Dense(256, activation='swish'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(128, activation='swish'),
    BatchNormalization(),
    Dropout(0.1),

    Dense(64, activation='swish'),
    Dense(1, activation='linear')
])

# 8. Compile Model
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.003)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# 9. Dynamic Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', patience=40, restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5
)

# 10. Train Model
history = model.fit(
    X_train_scaled, y_train,
    epochs=400,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# 11. Evaluate Predictions
y_pred = model.predict(X_test_scaled).flatten()

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"\n--- Model Results ---")
print(f"R2 Score: {r2:.4f} ({r2 * 100:.2f}%)")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - loss: 2107.0068 - mae: 38.1215 - val_loss: 528.6159 - val_mae: 20.2270 - learning_rate: 0.0030
Epoch 2/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 84.1319 - mae: 7.2436 - val_loss: 124.4388 - val_mae: 9.9589 - learning_rate: 0.0030
Epoch 3/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 43.9492 - mae: 5.2530 - val_loss: 55.1207 - val_mae: 6.2969 - learning_rate: 0.0030
Epoch 4/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 29.5030 - mae: 4.2953 - val_loss: 16.7861 - val_mae: 3.2854 - learning_rate: 0.0030
Epoch 5/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 23.1824 - mae: 3.8076 - val_loss: 9.7713 - val_mae: 2.3209 - learning_rate: 0.0030
Epoch 6/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 21.4714 - mae: 3.6716 - val_loss: 9.3181 - val_mae: 2.2084 - learning_rate: 0.0030
Epoch 7/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 19.1997 - mae: 3.4565 - val_loss: 10.1062 - val_mae: 2.2307 - learning_r